In [ ]:
import warnings
warnings.filterwarnings("ignore", category=Warning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    accuracy_score,
)

wineData = load_wine()
X = pd.DataFrame(wineData.data, columns=wineData.feature_names)
y = pd.Series(wineData.target)

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print(X.head())
print(y.value_counts().sort_index())

# First: separate training data from the remaining data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y,
)

# Second: split the remaining data into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp,
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)


In [ ]:
scaler = StandardScaler()

# Fit only on training data
X_train = scaler.fit_transform(X_train)

# Transform validation and test data using the same scaler
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

# Convert the data into PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.long)
y_val = torch.tensor(y_val.values, dtype=torch.long)
y_test = torch.tensor(y_test.values, dtype=torch.long)


In [ ]:
class ClassificationNN(nn.Module):
    def __init__(self, hidden_sizes):
        super().__init__()
        layers = []
        input_size = X_train.shape[1]
        output_size = len(wineData.target_names)

        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.Tanh())
            input_size = hidden_size

        layers.append(nn.Linear(input_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


In [ ]:
def train_model(model, X_train, y_train, X_val, y_val,
                epochs=500, learning_rate=0.01, weight_decay=0.0):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=learning_rate, weight_decay=weight_decay
    )
    train_losses, val_losses = [], []

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

        # Measure both losses using the updated model.
        model.eval()
        with torch.no_grad():
            train_losses.append(criterion(model(X_train), y_train).item())
            val_losses.append(criterion(model(X_val), y_val).item())

    return train_losses, val_losses


In [ ]:
# One CPU thread keeps these small experiments quick and reproducible.
torch.set_num_threads(1)
search_space = [
    ([2], 0.01, 0.0),
    ([8], 0.01, 0.0),
    ([16], 0.01, 0.0),
    ([64], 0.01, 0.0),
    ([16, 16], 0.01, 0.0),
    ([64, 64], 0.01, 0.0),
    ([16], 0.001, 0.0),
    ([16], 0.01, 0.01),
]

results = []
candidate_models = []
loss_histories = []

for hidden_sizes, lr, wd in search_space:
    torch.manual_seed(42)
    model = ClassificationNN(hidden_sizes)
    train_losses, val_losses = train_model(
        model, X_train, y_train, X_val, y_val,
        epochs=500, learning_rate=lr, weight_decay=wd,
    )
    results.append({
        "hidden_layers": str(hidden_sizes),
        "learning_rate": lr,
        "weight_decay": wd,
        "train_loss": train_losses[-1],
        "validation_loss": val_losses[-1],
    })
    candidate_models.append(model)
    loss_histories.append((train_losses, val_losses))

# Keep the original indices to retrieve the matching trained model.
results_df = pd.DataFrame(results).sort_values("validation_loss")
display(results_df)


In [ ]:
best_index = int(results_df.index[0])
best_config = results_df.loc[best_index]
lowest_train_index = int(results_df["train_loss"].idxmin())
print("Best validation configuration:")
print(best_config.to_string())
print("Lowest training loss also has lowest validation loss:",
      lowest_train_index == best_index)

train_losses, val_losses = loss_histories[best_index]
plt.figure(figsize=(7, 4))
plt.plot(train_losses, label="Training")
plt.plot(val_losses, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Loss for the selected configuration")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
best_model = candidate_models[best_index]
best_model.eval()
with torch.no_grad():
    test_predictions = best_model(X_test)
    y_pred = torch.argmax(test_predictions, dim=1)

test_accuracy = accuracy_score(y_test.numpy(), y_pred.numpy())
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy:.2%})")

cm = confusion_matrix(y_test.numpy(), y_pred.numpy(), labels=[0, 1, 2])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=wineData.target_names
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix - Test Dataset")
plt.tight_layout()
plt.show()

print(classification_report(
    y_test.numpy(), y_pred.numpy(), labels=[0, 1, 2],
    target_names=wineData.target_names, zero_division=0,
))


The selected model correctly classified 35 of 36 test observations, giving 97.22% test accuracy. It misclassified one class_1 wine as class_0.